# VoltWay Israel — Data Cleaning & Exploration

Final Project of the course -  Data Analyst. Data quality research and preprocessing

**Important:** datafiles located in the folder `data/`, containing source CSV. Output data will be placed in Database, and `clean_data/` folder


## 1. Loading and validating every csv

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")

dfs = {
    file.stem: pd.read_csv(file)
    for file in sorted(DATA_DIR.glob("*.csv"))
}

for name, df in dfs.items():
    print(f"{name}: {df.shape}")

columns_vehicle_registry_en: (22, 2)
ev_by_district: (90, 4)
lookup_districts: (27, 4)
lookup_fuel: (6, 2)
lookup_makers: (225, 5)
lookup_ownership: (5, 2)
lookup_wltp_body_type: (14, 2)
lookup_wltp_technology: (4, 2)
vehicle_registry_2024_2026: (861136, 22)
wltp_models_2024_2026: (19847, 94)


## 2. Checking data types, missing data and duplicates

In [2]:
for name, df in dfs.items():
    print(f"\n{'=' * 60}")
    print(f"TABLE: {name}")
    print(f"{'=' * 60}")

    print("\nColumns and types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum()[lambda x: x > 0])

    print("\nDuplicate rows:")
    print(df.duplicated().sum())

hebrew_columns = []




TABLE: columns_vehicle_registry_en

Columns and types:
column        str
meaning_en    str
dtype: object

Missing values:
Series([], dtype: int64)

Duplicate rows:
0

TABLE: ev_by_district

Columns and types:
baalut        str
mahoz_nm      str
nafa_nm       str
car_num     int64
dtype: object

Missing values:
mahoz_nm    2
nafa_nm     8
dtype: int64

Duplicate rows:
0

TABLE: lookup_districts

Columns and types:
mahoz_nm          str
nafa_nm           str
district_en       str
subdistrict_en    str
dtype: object

Missing values:
nafa_nm           5
subdistrict_en    5
dtype: int64

Duplicate rows:
0

TABLE: lookup_fuel

Columns and types:
sug_delek_nm    str
fuel_en         str
dtype: object

Missing values:
Series([], dtype: int64)

Duplicate rows:
0

TABLE: lookup_makers

Columns and types:
tozeret_cd               int64
tozeret_nm                 str
maker_en                   str
production_country_en      str
translation_note           str
dtype: object

Missing values:
producti

## 3. Registry: ID, production years, fuel types

In [3]:
vehicles = dfs["vehicle_registry_2024_2026"].copy()

print("\nTotal rows:", len(vehicles))
print("Unique IDs:", vehicles["mispar_rechev"].nunique())
print("Duplicate IDs:", vehicles["mispar_rechev"].duplicated().sum())

print("\nProduction years:")
print(vehicles["shnat_yitzur"].value_counts().sort_index())

print("\nFuel types:")
print(vehicles["sug_delek_nm"].value_counts(dropna=False))
hebrew_columns=[]
for col in vehicles.select_dtypes(include=["object", "string"]).columns:
    if vehicles[col].astype("string").str.contains(
        r"[\u0590-\u05FF]", na=False
    ).any():
        hebrew_columns.append(col)

print("Columns with Hebrew text:")
for col in hebrew_columns:
    print(col, "—", vehicles[col].nunique(), "unique values")


Total rows: 861136
Unique IDs: 861136
Duplicate IDs: 0

Production years:
shnat_yitzur
2024    272806
2025    299876
2026    288454
Name: count, dtype: int64

Fuel types:
sug_delek_nm
בנזין         538588
חשמל          163393
חשמל/בנזין    111528
דיזל           47432
חשמל/דיזל        165
גפמ"              30
Name: count, dtype: int64
Columns with Hebrew text:
tozeret_nm — 225 unique values
degem_manoa — 1241 unique values
baalut — 5 unique values
tzeva_rechev — 94 unique values
sug_delek_nm — 6 unique values


## 4. Missing pollution categories by fuel type

In [4]:
print(
    pd.crosstab(
        vehicles["sug_delek_nm"],
        vehicles["kvutzat_zihum"].isna()
    )
)

kvutzat_zihum   False   True 
sug_delek_nm                 
בנזין          538588       0
גפמ"               30       0
דיזל            47432       0
חשמל                0  163393
חשמל/בנזין     111528       0
חשמל/דיזל         165       0


## 5. Missing first registration months

In [5]:
missing_dates = vehicles[
    vehicles["moed_aliya_lakvish"].isna()
]

print(
    pd.crosstab(
        missing_dates["shnat_yitzur"],
        missing_dates["sug_delek_nm"],
        margins=True
    )
)

sug_delek_nm  בנזין  דיזל  All
shnat_yitzur                  
2024             28    26   54
2025            175    56  231
2026            106    42  148
All             309   124  433


## 6. Registration months format check

In [6]:
dates = vehicles["moed_aliya_lakvish"].dropna()

invalid_dates = dates[
    ~dates.str.fullmatch(r"\d{4}-(?:[1-9]|1[0-2])")
]

print("Invalid date formats:", len(invalid_dates))
print(invalid_dates.value_counts().head(20))

Invalid date formats: 0
Series([], Name: count, dtype: int64)


## 7. Registration date and quarter

In [7]:
vehicles["first_registration_date"] = pd.to_datetime(
    vehicles["moed_aliya_lakvish"],
    format="%Y-%m",
    errors="coerce"
)

vehicles["registration_quarter"] = (
    vehicles["first_registration_date"].dt.to_period("Q")
)

print(vehicles[[
    "moed_aliya_lakvish",
    "first_registration_date",
    "registration_quarter"
]].head(10))

print("Missing dates:",
      vehicles["first_registration_date"].isna().sum())

# Check monthly registrations
monthly = (
    vehicles.groupby(
        vehicles["first_registration_date"].dt.to_period("M")
    )
    .size()
    .sort_index()
)

print("Last 12 months:")
print(monthly.tail(12))

print("\nLast registration month:", monthly.index.max())
print("Last registration quarter:", monthly.index.max().asfreq("Q"))

last_full_quarter = pd.Period("2026Q3")

print("Last full quarter:", last_full_quarter)

  moed_aliya_lakvish first_registration_date registration_quarter
0            2025-10              2025-10-01               2025Q4
1             2026-9              2026-09-01               2026Q3
2             2026-2              2026-02-01               2026Q1
3             2026-1              2026-01-01               2026Q1
4             2026-1              2026-01-01               2026Q1
5             2026-3              2026-03-01               2026Q1
6             2026-6              2026-06-01               2026Q2
7            2025-11              2025-11-01               2025Q4
8             2026-1              2026-01-01               2026Q1
9             2024-6              2024-06-01               2024Q2
Missing dates: 433
Last 12 months:
first_registration_date
2025-12     9183
2026-01    44472
2026-02    27369
2026-03    29947
2026-04    22703
2026-05    28248
2026-06    30398
2026-07    27730
2026-08    29615
2026-09    24239
2026-10    22393
2026-11     1192
Freq: M, dt

## 8. Investigating missing trim level data

In [8]:
print(
    vehicles.loc[
        vehicles["ramat_gimur"].isna(),
        "tozeret_nm"
    ].value_counts().head(20)
)

tozeret_nm
דונגפנג סין       2002
איון סין          1149
פוטון סין          932
גי.איי.סי סין      813
וויה סין           700
אאודי גרמניה       529
ליפמוטור סין       177
ג'יי איי סי        157
באייק סין          136
מקסוס סין           97
אאודי ספרד          70
קוואן סין           50
פורשה גרמניה        38
פרארי               32
ג'יוואן סין         13
רנו צרפת             9
אס דבליו אמ          7
אסטון מרטין אנ       5
סנטרו סין            5
מודרן סין            2
Name: count, dtype: int64


## 9. Missimg safety data

In [9]:
print(
    pd.crosstab(
        vehicles["sug_delek_nm"],
        vehicles["ramat_eivzur_betihuty"].isna(),
        margins=True
    )
)

ramat_eivzur_betihuty   False   True     All
sug_delek_nm                                
בנזין                  519643  18945  538588
גפמ"                       29      1      30
דיזל                    40034   7398   47432
חשמל                   161229   2164  163393
חשמל/בנזין             111363    165  111528
חשמל/דיזל                 165      0     165
All                    832463  28673  861136


## 10. Other tech fields mising data

In [10]:
cols = ["zmig_ahori", "horaat_rishum"]

for col in cols:
    print(f"\n{col}")
    print(
        vehicles.loc[
            vehicles[col].isna(),
            ["tozeret_nm", "sug_delek_nm", "shnat_yitzur"]
        ].value_counts().head(10)
    )

print("\nVehicle colors:")
print("Unique color codes:", vehicles["tzeva_cd"].nunique())
print("Unique color names:", vehicles["tzeva_rechev"].nunique())

print(
    vehicles[["tzeva_cd", "tzeva_rechev"]]
    .drop_duplicates()
    .sort_values("tzeva_cd")
    .to_string(index=False)
)

color_translations = {
    1: "Pearl Green",
    2: "Pearl Black",
    3: "Pearl Blue",
    4: "Metallic Yellow",
    5: "Metallic Melange",
    6: "Dark Metallic Gray",
    7: "Metallic Silver Blue",
    8: "Metallic Copper",
    9: "Metallic Olive Green",
    10: "Black",
    11: "Metallic Black",
    13: "Metallic Olive",
    14: "Metallic Reddish",
    15: "Light Metallic Gray",
    16: "Bright Yellow",
    17: "Bright Red",
    19: "Matte Tangerine",
    20: "Gray",
    21: "Steel Gray",
    22: "Dark Gray",
    23: "Light Gray",
    24: "Bronze Gray",
    25: "Silver",
    26: "Platinum",
    27: "Metallic Gray",
    28: "Metallic Greenish",
    29: "Metallic Silver",
    30: "Blue",
    31: "Dark Blue",
    32: "Light Blue",
    33: "Sky Blue",
    34: "Turquoise",
    36: "Metallic Blue",
    37: "Metallic Blue Gray",
    38: "Metallic Charcoal Blue",
    39: "Metallic Sky Blue",
    40: "Green",
    41: "Light Green",
    42: "Dark Green",
    43: "Greenish",
    44: "Light Greenish",
    45: "Olive Green",
    46: "Metallic Green",
    47: "Metallic Burgundy",
    48: "Silver Green",
    49: "Metallic Gold Green",
    50: "Red",
    51: "Light Red",
    52: "Dark Wine Red",
    53: "Purple",
    54: "Burgundy",
    55: "Pink",
    56: "Metallic Red",
    57: "Eggplant",
    58: "Copper",
    59: "Metallic Rose",
    60: "Yellow",
    61: "Lemon Yellow",
    62: "Sahara",
    63: "Mustard",
    64: "Orange",
    65: "Gold",
    66: "Beige",
    67: "Dark Beige",
    68: "Cream",
    69: "Metallic Gold",
    70: "Brown",
    71: "Light Brown",
    72: "Dark Brown",
    73: "Golden",
    74: "Aqua Green",
    75: "Dark Purple",
    76: "Greenish Silver",
    77: "Classic Red",
    78: "Metallic Turquoise",
    79: "Sea Silver",
    80: "Ivory White",
    81: "Ivory",
    82: "Other",
    83: "Multicolor",
    84: "Light Metallic Turquoise",
    85: "Light Purple",
    86: "Millennium Silver",
    87: "Metallic Beige",
    88: "Metallic Coffee",
    91: "Crystal Blue",
    92: "Bronze",
    93: "Dark Silver",
    94: "Light Silver",
    95: "Turquoise Green",
    96: "Sea Green",
    97: "Metallic Bluish Silver",
    98: "Red Black",
    99: "Dark Metallic Silver"
}

vehicles["color_en"] = vehicles["tzeva_cd"].map(color_translations)

print("Missing translations:", vehicles["color_en"].isna().sum())
print("Unique translated colors:", vehicles["color_en"].nunique())


zmig_ahori
tozeret_nm      sug_delek_nm  shnat_yitzur
מרצדס בנץ ארהב  חשמל          2024            1
טויוטה צרפת     בנזין         2024            1
                              2025            1
סיטרואן צרפת    דיזל          2025            1
Name: count, dtype: int64

horaat_rishum
tozeret_nm      sug_delek_nm  shnat_yitzur
יונדאי ד. קורי  בנזין         2024            13
טויוטה יפן      בנזין         2024             9
סקודה צ'כיה     בנזין         2024             7
מרצדס בנץ גרמנ  דיזל          2024             6
קיה ד. קוריאה   בנזין         2024             5
רנו ד. קוריאה   בנזין         2024             4
טויוטה צרפת     בנזין         2024             4
סקודה צ'כיה     חשמל/בנזין    2024             4
טויוטה טורקיה   בנזין         2024             4
טויוטה ארהב"    בנזין         2024             3
Name: count, dtype: int64

Vehicle colors:
Unique color codes: 94
Unique color names: 94
 tzeva_cd      tzeva_rechev
        1        ירוק פנינה
        2        שחור פנינה
      

## 11. Fuel types 

In [11]:
print(dfs["lookup_fuel"].to_string(index=False))

sug_delek_nm                  fuel_en
       בנזין                   Petrol
        חשמל                 Electric
  חשמל/בנזין Hybrid (electric/petrol)
        דיזל                   Diesel
   חשמל/דיזל Hybrid (electric/diesel)
        גפמ"                      LPG


## 12. English fuel types merge

In [12]:
fuel = dfs["lookup_fuel"]

vehicles = vehicles.merge(
    fuel,
    on="sug_delek_nm",
    how="left",
    validate="many_to_one"
)

print("Rows after merge:", len(vehicles))
print("Missing translations:", vehicles["fuel_en"].isna().sum())
print(vehicles["fuel_en"].value_counts())

ownership = dfs["lookup_ownership"]

vehicles = vehicles.merge(
    ownership,
    on="baalut",
    how="left",
    validate="many_to_one"
)

print("Rows:", len(vehicles))
print("Missing translations:", vehicles["ownership_en"].isna().sum())

print(vehicles["ownership_en"].value_counts())

Rows after merge: 861136
Missing translations: 0
fuel_en
Petrol                      538588
Electric                    163393
Hybrid (electric/petrol)    111528
Diesel                       47432
Hybrid (electric/diesel)       165
LPG                             30
Name: count, dtype: int64
Rows: 861136
Missing translations: 0
ownership_en
Private    488357
Leasing    246318
Company     85053
Rental      29978
Dealer      11430
Name: count, dtype: int64


## 13. Car makers 

In [13]:
makers = dfs["lookup_makers"].copy()

print(makers.head(10).to_string(index=False))

print("\nDuplicate maker codes:")
print(makers["tozeret_cd"].duplicated().sum())

print("\nDuplicate code + name:")
print(
    makers.duplicated(
        subset=["tozeret_cd", "tozeret_nm"]
    ).sum()
)

 tozeret_cd     tozeret_nm   maker_en production_country_en translation_note
         19   אאודי גרמניה       Audi               Germany              NaN
         21  אאודי סלובקיה       Audi              Slovakia              NaN
         32  ב מ וו אנגליה        BMW        United Kingdom              NaN
         33    אופל גרמניה       Opel               Germany              NaN
         57     די אס צרפת         DS                France              NaN
         63    רנו סלובניה    Renault              Slovenia              NaN
         74 אלפא רומיאו_אי Alfa Romeo                 Italy              NaN
         76    אופל אנגליה       Opel        United Kingdom              NaN
         82 פולקסווגן פורט Volkswagen              Portugal              NaN
        101  איווקו איטליה      Iveco                 Italy              NaN

Duplicate maker codes:
0

Duplicate code + name:
0


## 14. Merge car makers and countries

In [14]:
vehicles = vehicles.merge(
    makers[["tozeret_cd", "maker_en", "production_country_en"]],
    on="tozeret_cd",
    how="left",
    validate="many_to_one"
)

print("Rows:", len(vehicles))
print("Missing makers:", vehicles["maker_en"].isna().sum())
print("Missing countries:", vehicles["production_country_en"].isna().sum())

print("\nTop 10 makers:")
print(vehicles["maker_en"].value_counts().head(10))

Rows: 861136
Missing makers: 0
Missing countries: 86723

Top 10 makers:
maker_en
Toyota        109084
Hyundai        84530
Kia            71937
Chery          67671
Skoda          57893
BYD            49019
Jaecoo         47735
MG             27797
Mazda          24395
Mitsubishi     19855
Name: count, dtype: int64


## 15. Car makers with no country info

In [15]:
missing_countries = vehicles[
    vehicles["production_country_en"].isna()
]

print(
    missing_countries.groupby(
        ["tozeret_cd", "maker_en", "tozeret_nm"]
    ).size().sort_values(ascending=False)
)

tozeret_cd  maker_en           tozeret_nm    
481         Hyundai            יונדאי ד. קורי    71571
1346        Lynk & Co          לינק אנד קו        7005
1566        KGM (KG Mobility)  קיי ג'י מובילי     5019
1381        FAW                אף איי דבליו       1544
1526        Jeep (Avenger)     ג'יפ אוונג'ר       1320
1213        JIC                ג'יי איי סי         157
1461        WMI                דבליו אם איי         42
770         Ferrari            פרארי                35
1569        SWM                אס דבליו אמ          30
dtype: int64


## 16. Untranslated countries info

In [16]:
print(
    makers.loc[
        makers["production_country_en"].isna(),
        ["tozeret_cd", "tozeret_nm", "maker_en", "translation_note"]
    ].to_string(index=False)
)

 tozeret_cd     tozeret_nm          maker_en                      translation_note
        481 יונדאי ד. קורי           Hyundai country part 'ד. קורי' not translated
        770          פרארי           Ferrari                                   NaN
       1213    ג'יי איי סי               JIC  transliteration - brand not verified
       1346    לינק אנד קו         Lynk & Co                                   NaN
       1381   אף איי דבליו               FAW                                   NaN
       1461   דבליו אם איי               WMI  transliteration - brand not verified
       1526   ג'יפ אוונג'ר    Jeep (Avenger)                                   NaN
       1566 קיי ג'י מובילי KGM (KG Mobility)                                   NaN
       1569    אס דבליו אמ               SWM                                   NaN


## 17. MIssing countries check with WLTP

In [17]:
wltp = dfs["wltp_models_2024_2026"]

missing_codes = makers.loc[
    makers["production_country_en"].isna(),
    "tozeret_cd"
]

print(
    wltp.loc[
        wltp["tozeret_cd"].isin(missing_codes),
        ["tozeret_cd", "tozeret_nm", "tozeret_eretz_nm"]
    ].drop_duplicates().to_string(index=False)
)

 tozeret_cd     tozeret_nm tozeret_eretz_nm
        481 יונדאי ד. קורי        ד. קוריאה
       1381   אף איי דבליו              סין
       1566 קיי ג'י מובילי        ד. קוריאה
       1569    אס דבליו אמ              סין
        770          פרארי           איטליה
       1213    ג'יי איי סי              סין
       1346    לינק אנד קו              סין
       1461   דבליו אם איי              סין
       1526   ג'יפ אוונג'ר            פולין


## 18. Check of WLTP data integrity

In [18]:
print(
    wltp.groupby("tozeret_cd")["tozeret_eretz_nm"]
    .nunique()
    .loc[missing_codes]
)

tozeret_cd
481     1
770     1
1213    1
1346    1
1381    1
1461    1
1526    1
1566    1
1569    1
Name: tozeret_eretz_nm, dtype: int64


## 19.  WLTP countries dictionary

In [19]:
country_map = (
    wltp[["tozeret_cd", "tozeret_eretz_nm"]]
    .drop_duplicates()
    .set_index("tozeret_cd")["tozeret_eretz_nm"]
)

print(country_map.loc[missing_codes])

tozeret_cd
481     ד. קוריאה
770        איטליה
1213          סין
1346          סין
1381          סין
1461          סין
1526        פולין
1566    ד. קוריאה
1569          סין
Name: tozeret_eretz_nm, dtype: str


## 20. Restoring Missing Production Countries

In [20]:

country_translations = {
    "ד. קוריאה": "South Korea",
    "סין": "China",
    "איטליה": "Italy",
    "פולין": "Poland"
}

missing_codes = makers.loc[
    makers["production_country_en"].isna(),
    "tozeret_cd"
]

country_lookup = (
    wltp[wltp["tozeret_cd"].isin(missing_codes)]
    [["tozeret_cd", "tozeret_eretz_nm"]]
    .drop_duplicates()
)

assert country_lookup["tozeret_cd"].is_unique
assert set(country_lookup["tozeret_cd"]) == set(missing_codes)

country_lookup["country_en"] = (
    country_lookup["tozeret_eretz_nm"]
    .map(country_translations)
)

assert country_lookup["country_en"].notna().all()

country_by_code = country_lookup.set_index(
    "tozeret_cd"
)["country_en"]

before = vehicles["production_country_en"].isna().sum()

vehicles["production_country_en"] = (
    vehicles["production_country_en"].fillna(
        vehicles["tozeret_cd"].map(country_by_code)
    )
)

makers["production_country_en"] = (
    makers["production_country_en"].fillna(
        makers["tozeret_cd"].map(country_by_code)
    )
)

after = vehicles["production_country_en"].isna().sum()

print("Missing before:", before)
print("Missing after:", after)
print("Restored:", before - after)


Missing before: 86723
Missing after: 0
Restored: 86723


## 21. Cleaning results validation

In [21]:
assert len(vehicles) == 861136
assert vehicles["mispar_rechev"].is_unique

assert vehicles["fuel_en"].notna().all()
assert vehicles["maker_en"].notna().all()
assert vehicles["production_country_en"].notna().all()

assert vehicles["first_registration_date"].isna().sum() == 433

print("All validation checks passed!")
print("Missing countries:", vehicles["production_country_en"].isna().sum())

All validation checks passed!
Missing countries: 0


## 22. Check districts data and WLTP integrity

In [22]:

districts = dfs["ev_by_district"].copy()

print("Total rows:", len(districts))

print("\nMissing values:")
print(districts.isna().sum())

print("\nRows with missing district or subdistrict:")
print(
    districts[
        districts["mahoz_nm"].isna() |
        districts["nafa_nm"].isna()
    ].to_string(index=False)
)

print(
    districts.groupby(
        "mahoz_nm",
        dropna=False
    )["car_num"].sum().sort_values(ascending=False)
)

print(
    dfs["lookup_districts"].to_string(index=False)
)


Total rows: 90

Missing values:
baalut      0
mahoz_nm    2
nafa_nm     8
car_num     0
dtype: int64

Rows with missing district or subdistrict:
baalut mahoz_nm nafa_nm  car_num
  פרטי      NaN     NaN      486
  חברה    הצפון     NaN      118
  חברה    המרכז     NaN       24
  חברה      NaN     NaN        9
  חברה  תל אביב     NaN        8
  חברה     חיפה     NaN        7
  חברה  ירושלים     NaN        3
  סוחר    הצפון     NaN        1
mahoz_nm
המרכז                 102525
תל אביב                42878
הצפון                  34889
הדרום                  32263
חיפה                   28819
ירושלים                15873
אזור יהודה ושומרון     11442
NaN                      495
Name: car_num, dtype: int64
          mahoz_nm      nafa_nm            district_en   subdistrict_en
           תל אביב      תל אביב               Tel Aviv         Tel Aviv
             המרכז    פתח תקווה                Central      Petah Tikva
             המרכז       רחובות                Central          Rehovot
 

## 23. District data check

In [23]:
lookup_districts = dfs["lookup_districts"]

print(
    lookup_districts.duplicated(
        subset=["mahoz_nm", "nafa_nm"]
    ).sum()
)

0


## 24. Merging english districts names

In [24]:
districts = districts.merge(
    lookup_districts,
    on=["mahoz_nm", "nafa_nm"],
    how="left",
    validate="many_to_one"
)

print("Rows:", len(districts))
print("Missing English districts:", districts["district_en"].isna().sum())
print("Missing English subdistricts:", districts["subdistrict_en"].isna().sum())

districts = districts.merge(
    dfs["lookup_ownership"],
    on="baalut",
    how="left",
    validate="many_to_one"
)

assert len(districts) == 90
assert districts["ownership_en"].notna().all()
print(districts["ownership_en"].value_counts())

Rows: 90
Missing English districts: 2
Missing English subdistricts: 8
ownership_en
Company    28
Private    23
Dealer     19
Leasing    12
Rental      8
Name: count, dtype: int64


## 25. WLTP validation

In [25]:
print("Rows:", len(wltp))

print("\nModel codes:")
print(wltp["degem_cd"].nunique())

print("\nDrive technologies:")
print(wltp["technologiat_hanaa_nm"].value_counts(dropna=False))

print(dfs["lookup_wltp_technology"].to_string(index=False))

body_types = dfs["lookup_wltp_body_type"]

print(body_types.to_string(index=False))

print("\nDuplicate body types:")
print(body_types["merkav"].duplicated().sum())

Rows: 19847

Model codes:
4358

Drive technologies:
technologiat_hanaa_nm
הנעה רגילה      8128
היברידי רגיל    5121
רכב חשמלי       4299
PLUG IN         2299
Name: count, dtype: int64
technologiat_hanaa_nm    drive_technology_en
           הנעה רגילה    Conventional engine
         היברידי רגיל Hybrid (self-charging)
            רכב חשמלי         Electric (BEV)
              PLUG IN         Plug-in hybrid
    merkav     body_type_en
  פנאי-שטח  SUV / crossover
      סדאן            Sedan
     הצ'בק        Hatchback
      קופה            Coupe
   קבריולט      Convertible
       MPV              MPV
     קומבי        Combi van
  משא אחוד Pickup / utility
   תא כפול       Double cab
    סטיישן    Station wagon
סגור/משלוח        Panel van
ואן/נוסעים    Passenger van
   תא בודד       Single cab
       שדה         Off-road

Duplicate body types:
0


## 26. English names for technology and body types

In [26]:
technology = dfs["lookup_wltp_technology"]

wltp = wltp.merge(
    technology,
    on="technologiat_hanaa_nm",
    how="left",
    validate="many_to_one"
)

print("Rows:", len(wltp))
print("Missing translations:", wltp["drive_technology_en"].isna().sum())

print(wltp["drive_technology_en"].value_counts())

wltp = wltp.merge(
    body_types,
    on="merkav",
    how="left",
    validate="many_to_one"
)

print("Rows:", len(wltp))
print("Missing body types:", wltp["body_type_en"].isna().sum())

print(wltp["body_type_en"].value_counts(dropna=False))

Rows: 19847
Missing translations: 0
drive_technology_en
Conventional engine       8128
Hybrid (self-charging)    5121
Electric (BEV)            4299
Plug-in hybrid            2299
Name: count, dtype: int64
Rows: 19847
Missing body types: 1
body_type_en
SUV / crossover     11222
Sedan                3250
Hatchback            1453
Coupe                1370
Convertible           734
MPV                   646
Combi van             310
Pickup / utility      225
Double cab            222
Station wagon         178
Panel van             134
Passenger van          77
Single cab             15
Off-road               10
NaN                     1
Name: count, dtype: int64


## 27. WLTP keys validation

In [27]:
print("Duplicate model codes:")
print(wltp["degem_cd"].duplicated().sum())

print("\nDuplicate maker + model:")
print(
    wltp.duplicated(
        subset=["tozeret_cd", "degem_cd"]
    ).sum()
)

print("\nUnique maker + model combinations:")
print(
    wltp[["tozeret_cd", "degem_cd"]]
    .drop_duplicates()
    .shape[0]
)

duplicates = wltp[
    wltp.duplicated(
        subset=["tozeret_cd", "degem_cd"],
        keep=False
    )
]

print(
    duplicates[
        [
            "tozeret_cd",
            "degem_cd",
            "degem_nm",
            "drive_technology_en",
            "body_type_en"
        ]
    ].head(20).to_string(index=False)
)

wltp_key = [
    "sug_degem",
    "tozeret_cd",
    "degem_cd",
    "shnat_yitzur"
]

print("\nFour-column key check:")
print("Total WLTP rows:", len(wltp))
print("Unique keys:", len(wltp[wltp_key].drop_duplicates()))

print(
    "Duplicate keys:",
    wltp.duplicated(subset=wltp_key).sum()
)

print("\nTechnology conflicts:")
print(
    wltp.groupby(wltp_key)["drive_technology_en"]
    .nunique(dropna=False)
    .value_counts()
    .sort_index()
)

print("\nBody type conflicts:")
print(
    wltp.groupby(wltp_key)["body_type_en"]
    .nunique(dropna=False)
    .value_counts()
    .sort_index()
)

matched = vehicles[wltp_key].merge(
    wltp[wltp_key],
    on=wltp_key,
    how="left",
    indicator=True,
    validate="many_to_one"
)

print(matched["_merge"].value_counts())
print(
    "Matched:",
    round((matched["_merge"] == "both").mean() * 100, 2),
    "%"
)

# Cars that didn't match WLTP
unmatched = vehicles.loc[
    matched["_merge"].eq("left_only").to_numpy()
]

print("Unmatched vehicles:", len(unmatched))

print("\nBy fuel:")
print(unmatched["fuel_en"].value_counts())

print("\nBy year:")
print(unmatched["shnat_yitzur"].value_counts().sort_index())

print("\nBy manufacturer:")
print(unmatched["maker_en"].value_counts().head(15))

Duplicate model codes:
15489

Duplicate maker + model:
5235

Unique maker + model combinations:
14612
 tozeret_cd  degem_cd       degem_nm drive_technology_en    body_type_en
         19      1383         8YARWY Conventional engine       Combi van
         21       179         4MN0N2 Conventional engine SUV / crossover
         74       256       949AXA2A Conventional engine SUV / crossover
        143       359           21CF      Electric (BEV) SUV / crossover
        143       359           21CF      Electric (BEV) SUV / crossover
        152       491        243.608      Electric (BEV) SUV / crossover
        187        69 GUN125L-DTTSXW Conventional engine      Double cab
        187        69 GUN125L-DTTSXW Conventional engine      Double cab
        187        71 GUN135L-DTTSXW Conventional engine      Double cab
        187        71 GUN135L-DTTSXW Conventional engine      Double cab
        187        72 GUN125L-DTTSXW Conventional engine      Double cab
        187        72 

In [28]:
# Cars that matched WLTP
vehicles = vehicles.merge(
    wltp[
        wltp_key + ["drive_technology_en", "body_type_en"]
    ],
    on=wltp_key,
    how="left",
    validate="many_to_one"
)

print("Vehicles:", len(vehicles))
print("Missing technology:", vehicles["drive_technology_en"].isna().sum())
print("Missing body type:", vehicles["body_type_en"].isna().sum())

assert len(vehicles) == 861136
assert vehicles["mispar_rechev"].is_unique

print(
    pd.crosstab(
        vehicles["fuel_en"],
        vehicles["drive_technology_en"],
        dropna=False
    ).to_string()
)



Vehicles: 861136
Missing technology: 297
Missing body type: 298
drive_technology_en       Conventional engine  Electric (BEV)  Hybrid (self-charging)  Plug-in hybrid  NaN
fuel_en                                                                                                   
Diesel                                  47009               0                     408               0   15
Electric                                    0          163329                       0               0   64
Hybrid (electric/diesel)                    0               0                       0             164    1
Hybrid (electric/petrol)                    0               0                       9          111459   60
LPG                                        22               0                       8               0    0
Petrol                                 300366               0                  238065               0  157


In [29]:
check = vehicles[
    (vehicles["fuel_en"] == "Petrol") &
    (vehicles["drive_technology_en"] == "Hybrid (self-charging)")
]

print(
    check.groupby(["maker_en", "degem_nm"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

maker_en  degem_nm      
Toyota    MXPJ10L-BHXNBW    22944
Hyundai   LS4DJ             14575
          HC811             14181
          HB811             12880
Jaecoo    GTB22             11205
Toyota    ZVG12L-KHXNBW      9337
Chery     BBDEG              8998
Toyota    AXAL52L-ANXPBW     8131
          ZVG12L-KHXGBW      7848
          MXPH11L-BHXNBW     7071
Kia       CP81EG             6565
Toyota    ZVG13L-RHXGBB      5487
MG        WP43               5353
Hyundai   JADD1              4718
Suzuki    JYAD2S             4616
Toyota    AXAL52L-ANXXBW     4557
MG        WS43               4435
Toyota    ZWE219L-DEXNBW     4109
Nissan    FCAF16             3800
Suzuki    UZ                 3520
dtype: int64


In [30]:
print(
    pd.crosstab(
        vehicles["fuel_en"],
        vehicles["drive_technology_en"],
        normalize="index"
    ).round(3).to_string()
)

drive_technology_en       Conventional engine  Electric (BEV)  Hybrid (self-charging)  Plug-in hybrid
fuel_en                                                                                              
Diesel                                  0.991             0.0                   0.009             0.0
Electric                                0.000             1.0                   0.000             0.0
Hybrid (electric/diesel)                0.000             0.0                   0.000             1.0
Hybrid (electric/petrol)                0.000             0.0                   0.000             1.0
LPG                                     0.733             0.0                   0.267             0.0
Petrol                                  0.558             0.0                   0.442             0.0


## 28. Cleaning Log

We checked all 10 CSV files.

1. We found no duplicate rows in the files. The vehicle IDs are also unique.

2. We found 163,393 missing values in kvutzat_zihum. All of them are electric cars. We decided to keep these values as they are.

3. We found 433 missing registration dates. These cars use petrol or diesel. We kept them, but we will not use them in the quarterly analysis.

4. We checked the date format. All existing dates are correct. We converted them to dates and added quarters.

5. We found 6,932 missing trim levels. We kept them because we do not need this field for our analysis.

6. We found 28,673 missing values in safety data. We kept them.

7. We found 4 missing values in zmig_ahori and 93 in horaat_rishum. We kept them.

8. We added English fuel names. All vehicles have a fuel translation.

9. We added English manufacturer names. All manufacturers were matched.

10. We found 86,723 cars without a production country. We checked the WLTP file and found the missing countries. We filled these values.

11. We found 2 missing districts and 8 missing subdistricts. We kept these values because we don't know the exact locations. No rows were deleted.

12. We found repeated maker and model codes in WLTP. Full rows are unique. We kept all records.


13. We translated the vehicle colors from Hebrew to English. We kept the original color names and codes.

14. We added English ownership names. All vehicles have a translation.

15. We matched 860,839 vehicles (99.97%) with WLTP using four columns: sug_degem, tozeret_cd, degem_cd and shnat_yitzur. We added drive technology and body type. We kept 297 unmatched vehicles without these values. Fuel types and drive technologies use different classifications, so we kept both.

16. We added English ownership names to the district table. All 90 rows have a translation. We kept the missing district and subdistrict values.

17. We found that November 2026 has only 1,192 registrations. The last quarter (2026Q4) is incomplete. We kept all records but will use 2026Q3 as the last full quarter for quarterly comparisons.

We started with 861,136 vehicles.
After cleaning, we still have 861,136 vehicles.
No vehicle rows were deleted.

## 29. Clean data export

In [31]:

OUTPUT_DIR = Path("clean_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

export_tables = dfs.copy()

export_tables["vehicle_registry_2024_2026"] = vehicles
export_tables["lookup_makers"] = makers
export_tables["wltp_models_2024_2026"] = wltp
export_tables["ev_by_district"] = districts

for name, df in export_tables.items():
    df.to_csv(
        OUTPUT_DIR / f"{name}.csv",
        index=False,
        encoding="utf-8-sig" # Hebrew characters support
    )

    print(f"Saved: {name} ({len(df)} rows)")


Saved: columns_vehicle_registry_en (22 rows)
Saved: ev_by_district (90 rows)
Saved: lookup_districts (27 rows)
Saved: lookup_fuel (6 rows)
Saved: lookup_makers (225 rows)
Saved: lookup_ownership (5 rows)
Saved: lookup_wltp_body_type (14 rows)
Saved: lookup_wltp_technology (4 rows)
Saved: vehicle_registry_2024_2026 (861136 rows)
Saved: wltp_models_2024_2026 (19847 rows)


## 30. Postrges DB connection

In [32]:
from sqlalchemy import create_engine, text, Date
from getpass import getpass

password = getpass("PostgreSQL password: ")
db_config = {
    "host": "127.0.0.1",
    "port": 5432,
    "user": "postgres",
    "password": password
}
# Connect to the default database
admin_engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={**db_config, "dbname": "postgres"},
    isolation_level="AUTOCOMMIT"
)
# Create database if needed
with admin_engine.connect() as conn:
    exists = conn.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :name"),
        {"name": "voltway_israel"}
    ).scalar()

    if not exists:
        conn.execute(text("CREATE DATABASE voltway_israel"))
        print("Database created: voltway_israel")
    else:
        print("Database already exists")

admin_engine.dispose()

# Connect to our project database

engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={**db_config, "dbname": "voltway_israel"}
)

with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database()"))
    print("Connected to:", result.scalar())

Database already exists
Connected to: voltway_israel


## 31. Database fill

In [33]:
vehicles_db = vehicles.copy()

date_columns = [
    "first_registration_date",
    "mivchan_acharon_dt",
    "tokef_dt"
]

for col in date_columns:
    vehicles_db[col] = pd.to_datetime(
        vehicles_db[col],
        errors="coerce"
    )

vehicles_db["registration_quarter"] = (
    vehicles_db["registration_quarter"].astype("string")
)

print(vehicles_db[date_columns].dtypes)
print(vehicles_db[date_columns].isna().sum())

vehicles_db.to_sql(
    "vehicle_registry",
    engine,
    schema="public",
    if_exists="replace",
    index=False,
    chunksize=5000,
    method="multi",
    dtype={col: Date() for col in date_columns}
)

first_registration_date    datetime64[us]
mivchan_acharon_dt         datetime64[us]
tokef_dt                   datetime64[us]
dtype: object
first_registration_date    433
mivchan_acharon_dt           0
tokef_dt                     0
dtype: int64


861136

## 32. Vehicles check before DB export

In [34]:
for col in date_columns:
    print(
        col,
        "Before:", vehicles[col].isna().sum(),
        "After:", vehicles_db[col].isna().sum()
    )

first_registration_date Before: 433 After: 433
mivchan_acharon_dt Before: 0 After: 0
tokef_dt Before: 0 After: 0


## 33. DB big export check

In [35]:
with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM public.vehicle_registry")
    ).scalar()

print("Rows in PostgreSQL:", count)

assert count == len(vehicles_db)

Rows in PostgreSQL: 861136


## 34. Export remaining tables

In [36]:
for name, df in export_tables.items():
    if name == "vehicle_registry_2024_2026":
        continue

    df.to_sql(
        name,
        engine,
        schema="public",
        if_exists="replace",
        index=False,
        chunksize=500
    )

    print(f"Uploaded: {name} ({len(df)} rows)")

# Create database indexes
indexes = [
    """
    CREATE UNIQUE INDEX IF NOT EXISTS idx_vehicle_id
    ON public.vehicle_registry (mispar_rechev)
    """,
    """
    CREATE INDEX IF NOT EXISTS idx_registration_date
    ON public.vehicle_registry (first_registration_date)
    """,
    """
    CREATE INDEX IF NOT EXISTS idx_drive_technology
    ON public.vehicle_registry (drive_technology_en)
    """
]

with engine.begin() as conn:
    for sql in indexes:
        conn.execute(text(sql))

print("Database indexes created!")


Uploaded: columns_vehicle_registry_en (22 rows)
Uploaded: ev_by_district (90 rows)
Uploaded: lookup_districts (27 rows)
Uploaded: lookup_fuel (6 rows)
Uploaded: lookup_makers (225 rows)
Uploaded: lookup_ownership (5 rows)
Uploaded: lookup_wltp_body_type (14 rows)
Uploaded: lookup_wltp_technology (4 rows)
Uploaded: wltp_models_2024_2026 (19847 rows)
Database indexes created!


## 35. Final validation

In [37]:
from sqlalchemy import inspect, text

inspector = inspect(engine)

with engine.connect() as conn:
    for name, df in export_tables.items():
        table = (
            "vehicle_registry"
            if name == "vehicle_registry_2024_2026"
            else name
        )

        assert inspector.has_table(table, schema="public")

        count = conn.execute(
            text(f'SELECT COUNT(*) FROM public."{table}"')
        ).scalar()

        assert count == len(df), f"{table}: row count mismatch"

        print(f"OK: {table} — {count} rows")

print("\nAll 10 tables validated!")

OK: columns_vehicle_registry_en — 22 rows
OK: ev_by_district — 90 rows
OK: lookup_districts — 27 rows
OK: lookup_fuel — 6 rows
OK: lookup_makers — 225 rows
OK: lookup_ownership — 5 rows
OK: lookup_wltp_body_type — 14 rows
OK: lookup_wltp_technology — 4 rows
OK: vehicle_registry — 861136 rows
OK: wltp_models_2024_2026 — 19847 rows

All 10 tables validated!


## 36. Final data notes
We finished preparing the data for the VoltWay Israel project.

All 10 tables were loaded into PostgreSQL database voltway_israel.

We checked the number of rows in all tables. Everything matches our prepared data.

Important notes:

The vehicle registry has 861,136 cars. Each car has a unique registration number. No cars were removed.

There are 163,393 electric cars in the registry. Hybrid cars are counted separately.

We have 433 cars without a first registration date. We cannot use them for quarterly analysis.

Some technical fields still have missing values. We decided to keep them because they are not important for our main questions.

We restored missing production countries using WLTP data.

The district table has 90 rows. Some locations are missing. We kept these records and did not guess their locations.

The WLTP table has 19,847 rows. Some maker and model codes are repeated, but there are no completely duplicated rows.

We should not join WLTP and vehicle registry using only maker and model codes. This could create extra rows and incorrect results.

We added English names for fuel types, manufacturers, ownership types, vehicle colors, districts, body types and drive technologies.

We added drive technology and body type from WLTP for 99.97% of vehicles. The fuel and technology classifications are different. We use drive_technology_en to distinguish BEV, plug-in hybrids, self-charging hybrids and conventional engines.

All prepared CSV files are saved in the clean_data folder. Original files are kept in the data folder.

Final result:

The data is ready for the next stage of analysis in PostgreSQL. We kept the original records and documented the known data limitations.

## 37. Loading charging data

We will add two files from the `current_data` folder:

- `charging-districts.csv`: charging points and privately owned EVs by district.
- `charging-cities.csv`: population and charging points by city.

We use `mahoz_nm` for the district name, as in `ev_by_district`. We can use this column to join the tables in SQL.

The vehicle data does not include the owner's city. We can calculate EVs per charging point by district, but not by city.


In [38]:
# Read the files. Keep the original files unchanged.
CURRENT_DIR = Path("current_data")

charging_districts = pd.read_csv(
    CURRENT_DIR / "charging-districts.csv",
    encoding="utf-8-sig"
)
charging_cities = pd.read_csv(
    CURRENT_DIR / "charging-cities.csv",
    encoding="utf-8-sig"
)

# Use English column names for SQL.
charging_districts = charging_districts.rename(columns={
    "מחוז": "mahoz_nm",
    "אתרי טעינה": "charging_sites",
    "עמדות": "charging_points",
    "מהירות": "fast_points_reported",
    "רכבים חשמליים פרטיים": "private_ev_reported",
    "רכבים לעמדה": "ev_per_point_reported",
    "מקור העמדות": "charging_source",
    "מקור הרכבים": "vehicle_source"
})

charging_cities = charging_cities.rename(columns={
    "עיר": "city_he",
    "slug": "city_slug",
    "מחוז": "mahoz_nm",
    'תושבים 2023 (למ"ס)': "population_2023",
    'אתרי טעינה עד 3 ק"מ מהמרכז': "sites_within_3km",
    "עמדות": "charging_points",
    "מהירות": "fast_points_reported",
    "עמדות ל-10,000 תושבים": "points_per_10000_reported",
    "מקור העמדות": "charging_source"
})

# The lookup contains several subdistricts for each district.
# Keep one row per district to avoid adding duplicate rows during the join.
district_names = (
    dfs["lookup_districts"][["mahoz_nm", "district_en"]]
    .dropna()
    .drop_duplicates()
)
assert district_names["mahoz_nm"].is_unique

charging_districts = charging_districts.merge(
    district_names,
    on="mahoz_nm",
    how="left",
    validate="many_to_one"
)
charging_cities = charging_cities.merge(
    district_names,
    on="mahoz_nm",
    how="left",
    validate="many_to_one"
)

# Extract the dates from the source descriptions and save them in separate columns.
for table in [charging_districts, charging_cities]:
    table["charging_snapshot_date"] = pd.to_datetime(
        table["charging_source"].str.extract(
            r"(\d{1,2}\.\d{1,2}\.\d{4})", expand=False
        ),
        format="%d.%m.%Y",
        errors="raise"
    )

charging_districts["vehicle_snapshot_date"] = pd.to_datetime(
    charging_districts["vehicle_source"].str.extract(
        r"(\d{1,2}\.\d{1,2}\.\d{4})", expand=False
    ),
    format="%d.%m.%Y",
    errors="raise"
)

print("Districts:", len(charging_districts))
print("Cities:", len(charging_cities))


Districts: 7
Cities: 143


## 38. Checking and saving the new tables

Each district must have one row. Each city must have a unique `city_slug`. We will check district names, dates and numeric values.


In [39]:
assert charging_districts["mahoz_nm"].is_unique
assert charging_cities["city_slug"].is_unique
assert set(charging_cities["mahoz_nm"]).issubset(
    set(charging_districts["mahoz_nm"])
)

charging_tables = {
    "charging_districts": charging_districts,
    "charging_cities": charging_cities
}

# Check that counts are present and not negative.
count_columns = {
    "charging_districts": [
        "charging_sites", "charging_points", "fast_points_reported",
        "private_ev_reported"
    ],
    "charging_cities": [
        "population_2023", "sites_within_3km", "charging_points",
        "fast_points_reported"
    ]
}

for name, table in charging_tables.items():
    assert not table.empty, f"{name}: the table is empty"
    assert table.notna().all().all(), f"{name}: missing values found"
    for col in count_columns[name]:
        table[col] = pd.to_numeric(table[col], errors="raise")
        assert table[col].ge(0).all(), f"{name}: negative values in {col}"
        assert table[col].mod(1).eq(0).all(), f"{name}: counts are not whole numbers in {col}"
        table[col] = table[col].astype("int64")
    assert table["fast_points_reported"].le(table["charging_points"]).all()

# Recalculate the reported ratios to check the values in the files.
assert charging_districts["charging_points"].gt(0).all()
assert charging_cities["population_2023"].gt(0).all()

calculated_ev_per_point = (
    charging_districts["private_ev_reported"]
    / charging_districts["charging_points"]
)
assert (
    charging_districts["ev_per_point_reported"] - calculated_ev_per_point
).abs().le(0.051).all()

calculated_points_per_10000 = (
    10000 * charging_cities["charging_points"]
    / charging_cities["population_2023"]
)
assert (
    charging_cities["points_per_10000_reported"] - calculated_points_per_10000
).abs().le(0.051).all()

# Save the prepared copies with the other clean data.
for name, table in charging_tables.items():
    table.to_csv(
        OUTPUT_DIR / f"{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print(f"Checked and saved: {name} — {len(table)} rows")


Checked and saved: charging_districts — 7 rows
Checked and saved: charging_cities — 143 rows


## 39. Uploading charging data to PostgreSQL

We use the `engine` connection created above. We upload only the two new tables. Running this cell again will replace these tables.


In [40]:
# Upload both tables in one transaction.
# If an error occurs, the changes in this transaction will be rolled back.
with engine.begin() as conn:
    for name, table in charging_tables.items():
        date_types = {
            col: Date()
            for col in table.columns
            if col.endswith("_snapshot_date")
        }
        table.to_sql(
            name,
            conn,
            schema="public",
            if_exists="replace",
            index=False,
            dtype=date_types
        )
        count = conn.execute(
            text(f'SELECT COUNT(*) FROM public."{name}"')
        ).scalar()
        assert count == len(table), f"{name}: the row count does not match"
        print(f"Uploaded: {name} — {count} rows")

    # Unique indexes prevent duplicate district names and city slugs.
    conn.execute(text("""
        CREATE UNIQUE INDEX idx_charging_district
        ON public.charging_districts (mahoz_nm)
    """))
    conn.execute(text("""
        CREATE UNIQUE INDEX idx_charging_city
        ON public.charging_cities (city_slug)
    """))

print("Both tables have been uploaded. They are ready for SQL.")


Uploaded: charging_districts — 7 rows
Uploaded: charging_cities — 143 rows
Both tables have been uploaded. They are ready for SQL.


## 40. How to join the tables

1. Use `mahoz_nm` to join the tables by district. First, add up `car_num` for each district in `ev_by_district`. Then join the result to the charging table. This keeps charging counts from being repeated.
2. `private_ev_reported` is the number of privately owned EVs in `charging_districts`. Divide it by `charging_points` to get EVs per charging point.
3. EV counts in `ev_by_district` and `private_ev_reported` may be different. Keep both counts. Do not add them together.
4. Charging counts are from 13 November 2025. EV counts in the new file are from 30 September 2026. The dates are different, so the ratio is only a rough guide.
5. City counts show charging points near the city centre. The same point may be counted for more than one city. Do not add city counts to get a district total.
6. The city file has no EV counts. We can compare charging points per 10,000 people. To compare EVs per charging point, we need EV counts for each city.
7. `fast_points_reported` comes from the `מהירות` column. The CSV does not say what charging power counts as fast. One charging site can have several charging points.


# Appendix. Stat tests.

We use `vehicles`, `districts` and `charging_districts` from above.
We do not read or save files here.

A1 is our main test. It answers the client's question about owner types. A2, A3 and A4 are extra checks.

We have already seen the counts. We use these tests to study the patterns, not to make new predictions.

We compare each p-value with 0.05. The tests show links, not causes.


## A1. Main test: owner type and EV ownership

**We expect:** EV shares differ by owner type. People and companies use cars for different needs.

**Why check this?** It can help us choose owner groups to study for partnerships.

- **H0:** All owner groups have the same EV share.
- **H1:** At least one group has a different EV share.

We use a chi-square test. EV means `Electric`. Non_EV means all other fuel types.


In [41]:
from scipy.stats import chi2_contingency, pearsonr

ALPHA = 0.05

# Count EVs and other cars by owner type.
ownership_counts = pd.crosstab(
    vehicles["ownership_en"],
    vehicles["fuel_en"].eq("Electric")
).rename(columns={False: "Non_EV", True: "EV"})

chi2, chi_p, chi_df, expected = chi2_contingency(
    ownership_counts, correction=False
)
assert expected.min() >= 5, "Expected counts are too small for this test."

# Show counts and EV shares.
ownership_result = ownership_counts.copy()
ownership_result["ev_share_pct"] = (
    100 * ownership_counts["EV"] / ownership_counts.sum(axis=1)
).round(2)

# Cramer's V shows how strong the link is: 0 is none, 1 is strongest.
# We have two groups: EV and Non_EV.
cramers_v = (chi2 / ownership_counts.to_numpy().sum()) ** 0.5

print(ownership_result.to_string())
print(f"\nChi-square: {chi2:.4f}")
print("Degrees of freedom:", chi_df)
print("p < 0.001" if chi_p < 0.001 else f"p = {chi_p:.4f}")
print(f"Cramer's V: {cramers_v:.4f}")
if chi_p < ALPHA:
    print("The result supports H1: EV shares differ by owner group.")
else:
    print("The result does not give enough support for H1: no clear difference was found.")


fuel_en       Non_EV      EV  ev_share_pct
ownership_en                              
Company        69492   15561         18.30
Dealer          7375    4055         35.48
Leasing       224019   22299          9.05
Private       367245  121112         24.80
Rental         29612     366          1.22

Chi-square: 34747.1408
Degrees of freedom: 4
p < 0.001
Cramer's V: 0.2009
The result supports H1: EV shares differ by owner group.


### What this tells us

EV shares show which owner groups have more EVs as a share of their cars. They do not show where people charge.

Dealer means a car dealer, not the final buyer.

Our cars were made in 2024-2026. They are not a random sample of all cars. With many cars, a small difference can give a low p-value. Look at the shares and Cramer's V too. Cars in one fleet may share the same buying decisions.


## A2. Extra test: private EVs and charging points

**We expect:** districts with more private EVs have more charging points. More EVs need places to charge.

**Why check this?** We want to see if charging points are more common where there are more EVs.

- **H0:** The correlation is zero.
- **H1:** The correlation is not zero.

We use Pearson correlation. One row is one district. We expect a positive link, but the test checks for a link in either direction.

Charging counts: **13 November 2025**.
EV counts: **30 September 2026**.

The dates are different. We cannot call this a test for 2025 alone.


In [42]:
# Use the district table from above.
corr_r, corr_p = pearsonr(
    charging_districts["private_ev_reported"],
    charging_districts["charging_points"]
)

print(charging_districts[[
    "district_en", "private_ev_reported", "charging_points"
]].to_string(index=False))
print("\nDistricts:", len(charging_districts))
print(f"Pearson r: {corr_r:.4f}")
print(f"p = {corr_p:.4f}")
if corr_p < ALPHA:
    print("The result supports H1: EV counts and charging points have a linear link.")
else:
    print("The result does not give enough support for H1: no clear linear link was found.")


           district_en  private_ev_reported  charging_points
              Northern                32098             1331
                 Haifa                24945              961
               Central                64462             1975
              Tel Aviv                28384             1599
             Jerusalem                13854              757
              Southern                29785             1351
Judea and Samaria Area                10739              486

Districts: 7
Pearson r: 0.9022
p = 0.0054
The result supports H1: EV counts and charging points have a linear link.


### What this tells us

A positive `r` means more EVs tend to go with more charging points. It does not mean there are enough charging points.

We have only seven districts. One large district can change the result a lot. They are not a random sample. Districts with more people may have more cars and charging points. A4 checks this using full district population totals. The p-value also needs the test assumptions to hold, so use it with care.

Also look at EVs per charging point. We can repeat the tests in Excel later.


## A3. Extra test: original EV counts and charging points

We use EV counts from `districts` and charging counts from `charging_districts`.

**We expect:** districts with more EVs have more charging points.

**Why check this?** We want to see if the original EV counts show the same pattern. We also compare EVs per point to find districts to study next.

- **H0:** The correlation is zero.
- **H1:** The correlation is not zero.

We expect a positive link. The test still checks for a link in either direction.

First, we add up EVs by district. Then we join the tables.
We also divide EVs by charging points to show EVs per point.
A3 uses all owner types. A2 uses only private EVs from the newer file.

The original file was downloaded on **8 November 2026**. We do not know the exact date of its EV counts. Charging counts are from **13 November 2025**. The dates do not match.


In [43]:
# Add up EVs by district. Keep the unknown group.
source_districts = (
    districts.groupby("mahoz_nm", dropna=False)["car_num"]
    .sum()
    .reset_index(name="source_ev_count")
)

# Join the totals to the charging table.
source_districts = source_districts.merge(
    charging_districts[["mahoz_nm", "district_en", "charging_points"]],
    on="mahoz_nm",
    how="left",
    validate="one_to_one"
)
# Divide EVs by charging points. Leave the result blank if the count is missing or zero.
source_districts["ev_per_charging_point"] = (
    source_districts["source_ev_count"]
    / source_districts["charging_points"].replace(0, float("nan"))
).round(2)

print(source_districts[[
    "district_en", "source_ev_count", "charging_points", "ev_per_charging_point"
]].fillna({"district_en": "Unknown"}).to_string(index=False))

# The test needs both counts. Keep unknown EVs in the table above.
matched_districts = source_districts.dropna(subset=["charging_points"])
# Find districts with no charging count. Then add up their EVs.
unmatched_districts = source_districts[
    source_districts["charging_points"].isna()
]
unmatched_ev = unmatched_districts["source_ev_count"].sum()

source_r, source_p = pearsonr(
    matched_districts["source_ev_count"],
    matched_districts["charging_points"]
)

print("\nDistricts in the test:", len(matched_districts))
print("EVs without matching charging data:", unmatched_ev)
print(f"Pearson r: {source_r:.4f}")
print(f"p = {source_p:.4f}")
if source_p < ALPHA:
    print("The result supports H1: EV counts and charging points have a linear link.")
else:
    print("The result does not give enough support for H1: no clear linear link was found.")


           district_en  source_ev_count  charging_points  ev_per_charging_point
Judea and Samaria Area            11442            486.0                  23.54
              Southern            32263           1351.0                  23.88
               Central           102525           1975.0                  51.91
              Northern            34889           1331.0                  26.21
                 Haifa            28819            961.0                  29.99
             Jerusalem            15873            757.0                  20.97
              Tel Aviv            42878           1599.0                  26.82
               Unknown              495              NaN                    NaN

Districts in the test: 7
EVs without matching charging data: 495
Pearson r: 0.8798
p = 0.0090
The result supports H1: EV counts and charging points have a linear link.


### What we can say

This can help us choose districts to study next. It cannot tell us which city needs a new site.

For that, we need EV counts by city and charging point locations from the same date. We also need to know how much people use the charging points.

The large vehicle table is still useful. It shows registrations over time, brands, countries and owner types. Registrations are not sales.

A3 has the same limits as A2: only seven districts and different dates. Both tests use the same charging counts, so they are not two separate proofs.


## A4. Extra test: allow for population

Bigger districts may have more EVs and more charging points just because more people live there.
We now check if EV counts and charging points are still linked after allowing for population. This is called partial correlation.

- **H0:** After allowing for population, the correlation is zero.
- **H1:** After allowing for population, the correlation is not zero.

We check both EV counts: private EVs from A2 and all EVs from A3.

The seven population values come from [CBS, Table 2.12: Localities and Population](https://www.cbs.gov.il/he/publications/DocLib/2026/2.ShnatonPopulation/st02_12x.xlsx), sheet `ST02-12x`, column B. They are for **31 December 2025**. The table was published on **8 October 2026**. The source uses thousands; we use people below. These totals include foreign residents. The Judea and Samaria value covers Israeli localities.

We use full district totals. We do not add up the 143 cities: they do not cover all places in each district. We do not change the course data or the charging files.


In [44]:
from scipy.stats import t

# CBS population by district, 31 December 2025. 
population_2025 = {
    "Jerusalem": 1359500,
    "Northern": 1599500,
    "Haifa": 1170200,
    "Central": 2435600,
    "Tel Aviv": 1551500,
    "Southern": 1561200,
    "Judea and Samaria Area": 525300
}

# Use the seven matched districts from A3. Add private EV counts from A2.
population_check = matched_districts.merge(
    charging_districts[["mahoz_nm", "private_ev_reported"]],
    on="mahoz_nm",
    validate="one_to_one"
)
population_check["population_2025"] = (
    population_check["district_en"].map(population_2025)
)
assert population_check["population_2025"].notna().all()

print(population_check[["district_en", "population_2025"]].to_string(index=False))

# Check private EVs and all EVs in the same way.
for ev_column in ["private_ev_reported", "source_ev_count"]:
    evs = population_check[ev_column]
    points = population_check["charging_points"]
    people = population_check["population_2025"]

    r_ev_points = pearsonr(evs, points).statistic
    r_ev_people = pearsonr(evs, people).statistic
    r_points_people = pearsonr(points, people).statistic

    # Remove the shared linear link with population.
    # We use partial Pearson correlation:
    # r_xy_z = (r_xy - r_xz * r_yz) / sqrt((1 - r_xz**2) * (1 - r_yz**2))
    # X = EV count, Y = charging point count, Z = population.
    # Bigger districts may have more EVs and more charging points.
    # This formula checks their linear link after allowing for population.
    adjusted_r = (
        (r_ev_points - r_ev_people * r_points_people)
        / ((1 - r_ev_people ** 2) * (1 - r_points_people ** 2)) ** 0.5
    )

    # One extra variable leaves n - 3 degrees of freedom.
    adjusted_df = len(population_check) - 3
    t_value = adjusted_r * (adjusted_df / (1 - adjusted_r ** 2)) ** 0.5
    adjusted_p = 2 * t.sf(abs(t_value), adjusted_df)

    print("\nEV count:", ev_column)
    print(f"Before allowing for population: r = {r_ev_points:.4f}")
    print(f"After allowing for population: r = {adjusted_r:.4f}")
    print(f"p = {adjusted_p:.4f}")
    if adjusted_p < ALPHA:
        print("The result supports H1: a linear link remains.")
    else:
        print("The result does not give enough support for H1 after allowing for population.")


           district_en  population_2025
Judea and Samaria Area           525300
              Southern          1561200
               Central          2435600
              Northern          1599500
                 Haifa          1170200
             Jerusalem          1359500
              Tel Aviv          1551500

EV count: private_ev_reported
Before allowing for population: r = 0.9022
After allowing for population: r = 0.3969
p = 0.4360
The result does not give enough support for H1 after allowing for population.

EV count: source_ev_count
Before allowing for population: r = 0.8798
After allowing for population: r = 0.3548
p = 0.4901
The result does not give enough support for H1 after allowing for population.


### What this tells us

The link is weaker after allowing for population. Neither test gives enough support for H1 at 0.05.

This suggests that district size matters. It does not prove that population explains the whole link, or that there is no link left.

There are only seven districts. Population is from December 2025, charging points are from November 2025, and the EV counts are from later data. The dates still do not match. The p-values also rely on a suitable linear model. This is a rough extra check. A1 is still our main test.


### Extra support: road traffic

EVs may charge outside the district where they are registered. Busy roads may also matter when choosing a charging site.

We saved [CBS Table 37](current_data/traffic_cbs_2026q1_table37.pdf) in `current_data`. [Original source](https://www.cbs.gov.il/he/publications/DocLib/2026/rt0126/h_print.pdf#page=58): Transport Statistics Quarterly No. 1, 2026, pages 58-59. Downloaded 21 November 2026.

The table has 35 road sections. The last count was in 2025 for 26 sections, in 2024 for eight, and in 2023 for one. It shows daily traffic and the dates of the counts. It includes all vehicle types, not just EVs.

We use it as extra support when choosing road corridors to study in priority districts. It is not a full traffic count for each district. Do not add section flows: the same car may pass several sections. The table does not show how many drivers will stop to charge.

We keep this separate from A1-A4. We do not change the course data or the charging tables.


### Final Appendix
We used the course data to study EV registrations, brands and owners. 
We added public charging data to compare EVs per charging point by district. 
The source dates are different, so this is a rough comparison. 
We would need EV counts by city and charging data from the same date to suggest specific sites.

From 2024Q1 to 2026Q3, BEV share fell and PHEV share grew. We cannot say that buyers switched from BEVs to PHEVs. The data does not show each buyer's choice.


### Data sources and links

- [Vehicle registry](https://data.gov.il/he/datasets/ministry_of_transport/private-and-commercial-vehicles): course file, downloaded 8 November 2026.
- [WLTP models](https://data.gov.il/he/datasets/ministry_of_transport/degem-rechev-wltp): course file, downloaded 8 November 2026.
- [EV counts by district](https://data.gov.il/he/datasets/ministry_of_transport/rechavim_hashmalim): course file. Also named as the source of private EV counts in `current_data`.
- [Charging stations](https://data.gov.il/he/datasets/ministry_of_transport/agg_charge_stations): the public source named inside our charging CSV files.
- [CBS population, 31 December 2025](https://www.cbs.gov.il/he/publications/DocLib/2026/2.ShnatonPopulation/st02_12x.xlsx): used in A4. Sheet `ST02-12x`, column B.

- [CBS road traffic, Table 37](https://www.cbs.gov.il/he/publications/DocLib/2026/rt0126/h_print.pdf#page=58): extra support for road corridor recommendations. [Local copy](current_data/traffic_cbs_2026q1_table37.pdf).

All **10 CSV files** in `data/` are loaded into `dfs` as separate DataFrames:

| File in `data/` | Role in the notebook |
|---|---|
| `vehicle_registry_2024_2026.csv` | Base vehicle DataFrame (`vehicles`). |
| `wltp_models_2024_2026.csv` | WLTP model DataFrame (`wltp`), joined to vehicles for model specifications and emissions. |
| `ev_by_district.csv` | EV counts by district, subdistrict and ownership (`districts`). |
| `lookup_fuel.csv` | English fuel labels joined to vehicles. |
| `lookup_ownership.csv` | English ownership labels joined to vehicles and district EV counts. |
| `lookup_makers.csv` | Maker and production-country labels joined to vehicles. |
| `lookup_districts.csv` | English district and subdistrict labels; also used for the charging tables. |
| `lookup_wltp_body_type.csv` | English body-type labels joined to WLTP models. |
| `lookup_wltp_technology.csv` | English drive-technology labels joined to WLTP models. |
| `columns_vehicle_registry_en.csv` | English descriptions of registry columns; loaded as a reference DataFrame, without a join to vehicles. |

The course supplied the six lookup tables and the column-description file. Full course notes are in `data/SOURCE.txt`.

The exact download URLs for our two charging summary files are not recorded. The city file also has no exact URL for its CBS 2023 population values. We should keep this gap clear. The official charging records downloaded later were only checked; they were not used to replace our files.

See [README: Data sources](README.md#data-sources) for dates, file history and the details of the official charging data check.
